# Train SOKE ASL 5K on Google Colab

Notebook nay dung theo flow hybrid: archive nam tren Google Drive, data/assets duoc copy va giai nen vao local `/content/SOKE_COLAB_DATA`, checkpoint ghi local truoc roi sync sang Drive moi epoch.

In [ ]:
# Sua 3 bien nay truoc khi chay.
GITHUB_REPO_URL = "https://github.com/KhoaLe1507/SOKE-Speech-to-SignLanguage-Realtim.git"
GITHUB_BRANCH = "main"
DRIVE_ROOT = "/content/drive/MyDrive/SOKE_COLAB"
LOCAL_ROOT = "/content/SOKE_COLAB_DATA"
LOCAL_ARCHIVE_ROOT = "/content/SOKE_COLAB_ARCHIVES"
LOCAL_RUN_ROOT = "/content/SOKE_COLAB_RUN"

REPO_DIR = "/content/SOKE"
CONFIG = "configs/soke_colab_asl_5k.yaml"
ASSETS_CONFIG = "configs/assets_colab.yaml"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!nvidia-smi

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)

subprocess.run([
    "git", "clone", "--branch", GITHUB_BRANCH,
    GITHUB_REPO_URL, REPO_DIR
], check=True)
os.chdir(REPO_DIR)
print("Repo dir:", os.getcwd())

In [ ]:
import sys
import subprocess

# Khong cai requirements.txt goc vi co bpy, phan train khong can Blender.
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", "requirements-colab.txt"
], check=True)

In [ ]:
import os
import shutil
import subprocess
from pathlib import Path

drive_root = Path(DRIVE_ROOT)
drive_archive_root = drive_root / "archives"
drive_token_cache = drive_root / "token_cache" / "TOKENS_how2sign_colab_5k"
local_root = Path(LOCAL_ROOT)
local_archive_root = Path(LOCAL_ARCHIVE_ROOT)
local_run_root = Path(LOCAL_RUN_ROOT)
data_root = local_root / "data"
deps_root = local_root / "deps"
pretrained_root = local_root / "pretrained"
for path in [local_root, local_archive_root, local_run_root, data_root, deps_root, pretrained_root, drive_root / "experiments", drive_root / "results", drive_token_cache]:
    path.mkdir(parents=True, exist_ok=True)

def first_existing(*paths):
    for path in paths:
        path = Path(path)
        if path.exists():
            return path
    return None

def copy_archive_to_local(archive_names):
    drive_archive = first_existing(*[drive_archive_root / name for name in archive_names])
    if drive_archive is None:
        return None
    local_archive = local_archive_root / drive_archive.name
    if (not local_archive.exists()) or local_archive.stat().st_size != drive_archive.stat().st_size:
        print("Copy archive to local:", drive_archive, "->", local_archive)
        shutil.copy2(drive_archive, local_archive)
    else:
        print("Local archive cache OK:", local_archive)
    return local_archive

def unpack_if_needed(expected_path, parent_dir, archive_names):
    expected_path = Path(expected_path)
    if expected_path.exists():
        print("Found:", expected_path)
        return
    archive = copy_archive_to_local(archive_names)
    if archive is None:
        print("Archive not found for:", expected_path)
        return
    print("Extracting", archive, "->", parent_dir)
    Path(parent_dir).mkdir(parents=True, exist_ok=True)
    if archive.suffix == ".zip":
        shutil.unpack_archive(str(archive), str(parent_dir))
    elif archive.name.endswith((".tar.gz", ".tgz")):
        subprocess.run(["tar", "-xzf", str(archive), "-C", str(parent_dir)], check=True)
    else:
        raise ValueError(f"Unsupported archive format: {archive}")

unpack_if_needed(data_root / "How2Sign", data_root, ["How2Sign.zip", "How2Sign.tar.gz", "How2Sign.tgz"])
unpack_if_needed(data_root / "stats", data_root, ["stats.zip", "stats.tar.gz", "stats.tgz"])
unpack_if_needed(deps_root / "mbart-h2s-csl-phoenix", deps_root, ["mbart-h2s-csl-phoenix.zip", "mbart-h2s-csl-phoenix.tar.gz", "mbart-h2s-csl-phoenix.tgz"])
unpack_if_needed(deps_root / "smpl_models", deps_root, ["smpl_models.zip", "smpl_models.tar.gz", "smpl_models.tgz"])
unpack_if_needed(deps_root / "t2m" / "glove", deps_root, ["t2m.tar.gz", "t2m.tgz", "t2m.zip"])

tokenizer_from_archives = copy_archive_to_local(["tokenizer.ckpt"])
tokenizer_target = pretrained_root / "tokenizer.ckpt"
if tokenizer_from_archives and tokenizer_from_archives.exists():
    if (not tokenizer_target.exists()) or tokenizer_target.stat().st_size != tokenizer_from_archives.stat().st_size:
        shutil.copy2(tokenizer_from_archives, tokenizer_target)
        print("Copied tokenizer checkpoint to:", tokenizer_target)

required_paths = [
    data_root / "How2Sign" / "train" / "poses",
    data_root / "How2Sign" / "train" / "re_aligned" / "how2sign_realigned_train_preprocessed_fps.csv",
    data_root / "How2Sign" / "val" / "poses",
    data_root / "How2Sign" / "val" / "re_aligned" / "how2sign_realigned_val_preprocessed_fps.csv",
    data_root / "stats" / "mean.pt",
    data_root / "stats" / "std.pt",
    pretrained_root / "tokenizer.ckpt",
    deps_root / "mbart-h2s-csl-phoenix" / "map_ids.pkl",
    deps_root / "mbart-h2s-csl-phoenix" / "pytorch_model.bin",
    deps_root / "smpl_models" / "smplx" / "SMPLX_NEUTRAL.npz",
    deps_root / "smpl_models" / "smplx" / "SMPLX_to_J14.pkl",
    deps_root / "smpl_models" / "smplx_vert_segmentation.json",
    deps_root / "t2m" / "glove" / "our_vab_data.npy",
]
missing = [str(p) for p in required_paths if not p.exists()]
if missing:
    raise FileNotFoundError("Missing required Colab files after local extraction:\n" + "\n".join(missing))

Path("deps").mkdir(exist_ok=True)

def force_symlink(src, dst):
    dst = Path(dst)
    if dst.is_symlink() or dst.exists():
        if dst.is_dir() and not dst.is_symlink():
            shutil.rmtree(dst)
        else:
            dst.unlink()
    os.symlink(src, dst, target_is_directory=True)

force_symlink(deps_root / "mbart-h2s-csl-phoenix", "deps/mbart-h2s-csl-phoenix")
force_symlink(deps_root / "smpl_models", "deps/smpl_models")
force_symlink(deps_root / "t2m", "deps/t2m")

print("Hybrid local data layout OK")

In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path

local_code_root = Path(LOCAL_ROOT) / "data" / "How2Sign" / "TOKENS_how2sign_colab_5k"
local_token_dir = local_code_root / "how2sign"
drive_code_root = Path(DRIVE_ROOT) / "token_cache" / "TOKENS_how2sign_colab_5k"
drive_token_dir = drive_code_root / "how2sign"

def count_tokens(path):
    return len(list(path.glob("*.npy"))) if path.exists() else 0

local_token_count = count_tokens(local_token_dir)
drive_token_count = count_tokens(drive_token_dir)
print("Local token files:", local_token_count)
print("Drive token cache files:", drive_token_count)

if local_token_count < 5000 and drive_token_count >= 5000:
    print("Restoring token cache from Drive to local disk")
    shutil.copytree(drive_code_root, local_code_root, dirs_exist_ok=True)
    local_token_count = count_tokens(local_token_dir)

if local_token_count < 5000:
    cmd = [
        sys.executable, "-m", "scripts.get_motion_code",
        "--cfg", CONFIG,
        "--cfg_assets", ASSETS_CONFIG,
        "--nodebug",
        "--device", "0",
        "--use_gpus", "0",
    ]
    subprocess.run(cmd, check=True)
    local_token_count = count_tokens(local_token_dir)

if local_token_count < 5000:
    raise RuntimeError(f"Token cache is incomplete: {local_token_count}/5000 files in {local_token_dir}")

if count_tokens(drive_token_dir) < local_token_count:
    print("Saving local token cache back to Drive")
    drive_code_root.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(local_code_root, drive_code_root, dirs_exist_ok=True)
else:
    print("Token cache already exists; skip tokenization.")

In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path

local_exp_dir = Path(LOCAL_RUN_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_5K"
local_ckpt_dir = local_exp_dir / "checkpoints"
local_last_ckpt = local_ckpt_dir / "last.ckpt"
drive_exp_dir = Path(DRIVE_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_5K"
drive_last_ckpt = drive_exp_dir / "checkpoints" / "last.ckpt"

if drive_last_ckpt.exists():
    local_ckpt_dir.mkdir(parents=True, exist_ok=True)
    should_restore = (
        (not local_last_ckpt.exists())
        or local_last_ckpt.stat().st_size != drive_last_ckpt.stat().st_size
        or drive_last_ckpt.stat().st_mtime > local_last_ckpt.stat().st_mtime
    )
    if should_restore:
        print("Restore checkpoint from Drive to local:", drive_last_ckpt, "->", local_last_ckpt)
        shutil.copy2(drive_last_ckpt, local_last_ckpt)
    else:
        print("Keep local checkpoint because it is newer or equal:", local_last_ckpt)

cmd = [
    sys.executable, "-m", "train",
    "--cfg", CONFIG,
    "--cfg_assets", ASSETS_CONFIG,
    "--nodebug",
    "--device", "0",
    "--use_gpus", "0",
]

if local_last_ckpt.exists():
    print("Resume from local checkpoint:", local_last_ckpt)
    cmd += ["--resume", str(local_exp_dir)]
else:
    print("Start a new training run.")

subprocess.run(cmd, check=True)

In [ ]:
from pathlib import Path

local_ckpt_dir = Path(LOCAL_RUN_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_5K" / "checkpoints"
drive_ckpt_dir = Path(DRIVE_ROOT) / "experiments" / "mgpt" / "SOKE_COLAB_ASL_5K" / "checkpoints"

print("Local checkpoint dir:", local_ckpt_dir)
for path in sorted(local_ckpt_dir.glob("*.ckpt")):
    print("local", path.name, f"{path.stat().st_size / (1024 ** 3):.2f} GiB")

print("Drive checkpoint dir:", drive_ckpt_dir)
for path in sorted(drive_ckpt_dir.glob("*.ckpt")):
    print("drive", path.name, f"{path.stat().st_size / (1024 ** 3):.2f} GiB")